In [ ]:

# # Integrated Clustering and Marco’s Edge Analysis
# 
# In this notebook cell we first load trajectory data (from play and expert HDF5 files), resample and cluster them.
# For each expert cluster (as determined by our dynamic scoring), we then:
# 
# 1. Gather all the raw 3D positions (states) from that cluster.
# 2. Select a representative demo (the first trajectory in that cluster).
# 3. Run Marco’s HDBSCAN–based analysis on the cluster (including centroid/epsilon calculation, prediction, and edge splitting).
# 4. Plot the “edges” with Plotly.
#
# Adjust file paths, parameters, and thresholds as necessary.


import os
import numpy as np
import h5py
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # For 3D plotting
import hdbscan
from sklearn.cluster import MeanShift, estimate_bandwidth
from scipy.spatial.transform import Rotation as R
import plotly.express as px
import plotly.graph_objects as go

# -----------------------------------------------
# Marco's Algorithm Functions 
# -----------------------------------------------
def marco_hdbscan(X, min_cluster_size=200):
    """
    Applies HDBSCAN clustering with the option to store cluster centroids.
    """
    clustering = hdbscan.HDBSCAN(min_cluster_size=min_cluster_size, 
                                 store_centers="centroid").fit(X)
    return clustering

def hdbscan_predict(X, centroids, eps):
    """
    Given a set of centroids and their corresponding epsilons, assign each point in X
    to the closest centroid if within eps distance. Otherwise, label as -1.
    """
    labels = -1 * np.ones(X.shape[0], dtype=int)
    for i, center in enumerate(centroids):
        # Compute distances from all points to the current center
        distances = np.linalg.norm(X - center, axis=1)
        # For simplicity, assign the point with the minimum distance to this cluster
        min_idx = np.argmin(distances)
        if distances[min_idx] <= eps[i]:
            labels[min_idx] = i
    return labels

def calculate_centroids(X, labels):
    """
    For each cluster (ignoring label -1), compute the centroid (mean of the points).
    """
    centroids = np.empty((0, X.shape[1]))
    for label in set(labels):
        if label == -1:
            continue
        mask = (labels == label)
        cluster_points = X[mask]
        centroid = np.mean(cluster_points, axis=0, keepdims=True)
        centroids = np.concatenate((centroids, centroid), axis=0)
    return centroids

def calculate_eps(X, centroids, label_set, labels):
    """
    For each cluster, determine the maximum distance from its points to the centroid.
    """
    epsilons = []
    for label in label_set:
        if label == -1:
            continue
        mask = (labels == label)
        cluster_points = X[mask]
        distances = np.linalg.norm(cluster_points - centroids[label], axis=1)
        epsilons.append(np.max(distances))
    return epsilons

def split_edges(X, labels):
    """
    Split the 3D trajectory X into segments (edges) based on changes in predicted labels.
    Returns a dictionary where keys are edge indices and values are (n_points, 3) arrays.
    """
    assert len(X) == len(labels)
    labels = np.array(labels)
    indices = list(np.argwhere(labels >= 0).flatten())
    # Insert start and end indices to segment the data
    indices.insert(0, 0)
    indices.append(len(labels))
    res = {}
    edge = 0
    for i in range(len(indices) - 1):
        res[edge] = X[indices[i]:indices[i+1], :]
        edge += 1
    return res

def plot_edges(edges):
    """
    Create a 3D Plotly plot for each edge (segment) using different colors.
    """
    colors = px.colors.qualitative.Plotly
    fig = go.Figure()
    for i, (edge, points) in enumerate(edges.items()):
        color = colors[i % len(colors)]
        fig.add_trace(go.Scatter3d(
            x=points[:, 0],
            y=points[:, 1],
            z=points[:, 2],
            mode='markers',
            marker=dict(color=color, size=8),
            name=f"Edge {edge}"
        ))
    fig.update_layout(title="Edges from Marco's HDBSCAN Analysis",
                      scene=dict(xaxis_title="X", yaxis_title="Y", zaxis_title="Z"))
    fig.show()
    
    # fig.write_html('cluster_edges.html')
    return fig

# -----------------------------------------------
# Our Clustering Algorithm Functions
# -----------------------------------------------
def load_trajectories(file_path):
    """
    Loads complete trajectories from the given HDF5 file.
    Each demonstration (demo) is assumed to have an 'obs/states' dataset.
    Returns:
        A list of np.arrays, each of shape (n_steps, state_dim).
    """
    trajectories = []
    with h5py.File(file_path, 'r') as hdf:
        data_group = hdf['data']
        for demo_key in data_group:
            demo_group = data_group[demo_key]
            if 'obs' in demo_group and 'states' in demo_group['obs']:
                traj = demo_group['obs']['states'][:]
                if traj.shape[0] > 0:
                    trajectories.append(traj)
    return trajectories

def choose_resample_length(trajectories):
    """
    Chooses a target length to which all trajectories will be resampled.
    Here we use the median length of all trajectories.
    """
    lengths = [traj.shape[0] for traj in trajectories]
    return int(np.median(lengths))

def resample_trajectory(traj, new_length):
    """
    Resamples a trajectory to have exactly 'new_length' timesteps via linear interpolation.
    """
    original_length = traj.shape[0]
    state_dim = traj.shape[1]
    old_indices = np.arange(original_length)
    new_indices = np.linspace(0, original_length - 1, new_length)
    new_traj = np.zeros((new_length, state_dim))
    for d in range(state_dim):
        new_traj[:, d] = np.interp(new_indices, old_indices, traj[:, d])
    return new_traj

def extract_features(traj):
    """
    Flattens the trajectory into a single 1D feature vector.
    """
    return traj.flatten()

def compute_cluster_entropy(cluster_features):
    """
    Computes a simple differential entropy assuming a multivariate Gaussian distribution.
    """
    cov = np.cov(cluster_features, rowvar=False)
    d = cluster_features.shape[1]
    sign, logdet = np.linalg.slogdet(cov)
    if sign <= 0:
        cov += np.eye(d) * 1e-6
        sign, logdet = np.linalg.slogdet(cov)
    entropy = 0.5 * (d * np.log(2 * np.pi * np.e) + logdet)
    return entropy

def extract_hotspots(points, quantile=0.2):
    """
    Uses MeanShift clustering (via kernel density estimation) to identify hotspots.
    """
    if points.shape[0] == 0:
        return np.empty((0, points.shape[1]))
    bandwidth = estimate_bandwidth(points, quantile=quantile, n_samples=len(points))
    ms = MeanShift(bandwidth=bandwidth, bin_seeding=True)
    ms.fit(points)
    return ms.cluster_centers_

# -----------------------------------------------
# Integration: Marco's Analysis on Individual Clusters
# -----------------------------------------------
def analyze_cluster_with_marco(cluster_trajs, min_cluster_size_frac=0.1):
    """
    For a given list of trajectories in a cluster, this function:
      - Combines raw 3D state points from all trajectories.
      - Selects a representative trajectory (the first one) as a demo.
      - Computes an appropriate min_cluster_size (based on a fraction of the total points).
      - Applies Marco's HDBSCAN-based analysis and edge-splitting.
      - Plots the resulting edges using Plotly.
    """
    # Assume each trajectory has at least 3 columns (for 3D positions)
    # Combine all trajectories' positions (only first 3 columns) into one array.
    X = np.vstack([traj[:, :3] for traj in cluster_trajs])
    # Compute the average trajectory over all trajectories in the cluster
    cluster_array = np.stack(cluster_trajs, axis=0)
    avg_traj = np.mean(cluster_array, axis=0)  # shape: (T, state_dim)
    one_demo = avg_traj[:, :3]  # Use the average trajectory's 3D positions
    
    # Determine min_cluster_size based on a fraction of the total points
    min_cluster_size = max(5, int(min_cluster_size_frac * X.shape[0]))
    print(f"Running Marco's HDBSCAN on a cluster with {X.shape[0]} points, min_cluster_size = {min_cluster_size}")
    
    clustering = marco_hdbscan(X, min_cluster_size=min_cluster_size)
    labels = clustering.labels_
    # For safety, recalc centroids using our helper (Marco's HDBSCAN with store_centers may be used directly)
    centroids = clustering.centers_ if hasattr(clustering, "centers_") else calculate_centroids(X, labels)
    
    # Calculate epsilons for each cluster (here we use the order of centroids)
    epsilons = calculate_eps(X, centroids, set(labels), labels)
    # Predict labels on the representative demo using our hdbscan_predict function.
    predicted_labels = hdbscan_predict(one_demo, centroids, epsilons)
    edges = split_edges(one_demo, predicted_labels)
    
    print(f"Number of edges identified: {len(edges)}")
    fig = plot_edges(edges)
    return fig

# -----------------------------------------------
# Main Workflow: Load Data, Cluster, and Apply Marco's Analysis on Expert Clusters
# -----------------------------------------------
# Set your file paths (adjust as needed)
hdf_path_play   = '/gscratch/stf/roboviz/play_pushing.hdf5'
hdf_path_expert = '/gscratch/stf/roboviz/expert_lampshade2_demos.hdf5'

# Load trajectories from both datasets
trajectories_play = load_trajectories(hdf_path_play)
trajectories_expert = load_trajectories(hdf_path_expert)
trajectories = trajectories_play + trajectories_expert
print(f"Total trajectories loaded: {len(trajectories)}")

# Resample all trajectories to a common length (using the median length)
target_length = choose_resample_length(trajectories)
print("Target resample length (median):", target_length)
all_traj_resampled = [resample_trajectory(traj, new_length=target_length) for traj in trajectories]

# Extract features for clustering (flatten each trajectory)
features = np.array([extract_features(traj) for traj in all_traj_resampled])

# Cluster using HDBSCAN (here we use low min_cluster_size for demonstration)
clusterer = hdbscan.HDBSCAN(min_cluster_size=5, min_samples=2)
cluster_labels = clusterer.fit_predict(features)
unique_labels = np.unique(cluster_labels)
print("HDBSCAN cluster labels:", unique_labels, "( -1 indicates outliers )")

# Compute entropy and sizes for valid clusters
cluster_entropies = {}
valid_labels = [lbl for lbl in unique_labels if lbl != -1]
for lbl in valid_labels:
    cluster_features = features[cluster_labels == lbl]
    entropy = compute_cluster_entropy(cluster_features)
    cluster_entropies[lbl] = entropy
    print(f"Cluster {lbl} entropy: {entropy:.4f}")
    
cluster_sizes = {lbl: np.sum(cluster_labels == lbl) for lbl in valid_labels}
print("Cluster sizes:", cluster_sizes)

total_valid = sum(cluster_sizes[lbl] for lbl in valid_labels)
cluster_scores = {lbl: (-cluster_entropies[lbl]) * (cluster_sizes[lbl] / total_valid)
                  for lbl in valid_labels}
for lbl in valid_labels:
    print(f"Cluster {lbl} score: {cluster_scores[lbl]:.4f}")

# Determine expert clusters based on dynamic threshold (median score)
scores_array = np.array(list(cluster_scores.values()))
dynamic_threshold = np.median(scores_array)
expert_clusters = [lbl for lbl, score in cluster_scores.items() if score >= dynamic_threshold]
print("Dynamically identified expert clusters:", expert_clusters)

# -------------------------------
# Visualization of Trajectories (Optional Overview Plot)
# -------------------------------
fig_overview = plt.figure(figsize=(12, 10))
ax_overview = fig_overview.add_subplot(111, projection='3d')
n_clusters = len(valid_labels)
cmap = plt.get_cmap('viridis', n_clusters if n_clusters > 0 else 1)
label_to_color_idx = {lbl: i for i, lbl in enumerate(valid_labels)}
legend_labels = {"Play Data": False}
for expert in expert_clusters:
    legend_labels[f"Expert Cluster {expert}"] = False

for traj, lbl in zip(all_traj_resampled, cluster_labels):
    if lbl in expert_clusters:
        color_idx = label_to_color_idx[lbl]
        color = cmap(color_idx)
        lw = 2.5
        label_str = f"Expert Cluster {lbl}"
    else:
        color = "gray"
        lw = 1.5
        label_str = "Play Data"
    if not legend_labels[label_str]:
        ax_overview.plot(traj[:, 0], traj[:, 1], traj[:, 2], color=color, alpha=0.7, lw=lw, label=label_str)
        legend_labels[label_str] = True
    else:
        ax_overview.plot(traj[:, 0], traj[:, 1], traj[:, 2], color=color, alpha=0.7, lw=lw)
ax_overview.set_title("Overview: Trajectories and Expert Clusters")
ax_overview.set_xlabel("X")
ax_overview.set_ylabel("Y")
ax_overview.set_zlabel("Z")
ax_overview.legend(loc='best')
plt.tight_layout()
plt.show()

# -------------------------------
# Apply Marco's Algorithm on Each Expert Cluster Separately
# -------------------------------
# For each expert cluster, gather the trajectories and run the integrated analysis.
for lbl in expert_clusters:
    # Get indices of trajectories belonging to this expert cluster
    cluster_indices = np.where(cluster_labels == lbl)[0]
    cluster_trajs = [all_traj_resampled[i] for i in cluster_indices]
    print(f"\n=== Analyzing Expert Cluster {lbl} with {len(cluster_trajs)} trajectories ===")
    fig = analyze_cluster_with_marco(cluster_trajs, min_cluster_size_frac=0.1)
    # Each fig is interactive. In Jupyter you can interact with the Plotly output.

# %% [markdown]
# ## Next Steps
#
# You can further refine the integration by, for example, storing the separate plots,
# performing additional analysis on the edges, or comparing expert versus play clusters.